<div style="background:#C1ABA6">
<div style="font-size: xx-large ; font-weight: 900 ;  padding-top: 100px ; color: rgba(0 , 0 , 0 , 0.8); line-height: 100%"; align="center">Makroseizmički intenziteti</div>
    
---
<div align="center"> Vježbe iz Seizmologije I </div>
<div align="center"> ak. god. 2026./2027. </div>
<div align="center"> dr.sc. Katarina Zailac </div>

---
</div>

Procijenjene makroseizmičke intenzitete za potres s prethodnog sata trebamo prikazati na karti. Datoteku `intenziteti.csv` *uploadajte* u direktorij `data`.

In [ ]:
# podaci o hipocentru
lat = 
lon = 
h = 
i0 = 

### Učitavanje podataka

Podatke ćemo učitati korištenjem funkcije [`read_csv`](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) iz biblioteke [pandas](https://pandas.pydata.org/docs/).

In [ ]:
import pandas as pd

df = pd.read_csv("data/intenziteti.csv")

print(df)

Podaci se učitaju u tablicu tipa [`dataframe`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html). Do pojedinog stupca se dolazi jednostavno pozivanjem varijable u koju je `dataframe` spremljen i imena stupca. Na primjer, ako želimo doći do imena mjesta koja su navedena u tablici, to ćemo napraviti kao: 

In [ ]:
mjesta = df["Mjesto"]

print(mjesta)

Spremimo intenzitete u varijablu `intensity`. Želimo da su intenziteti poredani od najmanjeg prema najvećem, i da nema ponavljanja vrijednosti!

In [ ]:
intensity = df["Intenzitet"].drop_duplicates().sort_values()

print(intensity.to_list())

### Prikaz intenziteta na karti

Nacrtajte kartu RH s bijelom podlogom i označite državne granice!

In [ ]:
import pygmt


Sada ćemo ucrtati procijenjene intenzitete!

Najprije trebamo definirati *colorscale* koji ćemo koristiti za prikaz intenziteta. Za to koristimo funkciju [`pygmt.makecpt`](https://www.pygmt.org/latest/api/generated/pygmt.makecpt.html). 

Koristit ćemo pred-definirani *color scale* [gmt/seis](https://docs.generic-mapping-tools.org/6.5/reference/cpts.html#built-in-color-palette-tables-cpt), koji unosimo kao parametar `cmap`. Želimo da nam boje idu u obrnutom redosljedu od onoga kako je definirano, pa ćemo staviti parametar `reverse=True`.

Granice veličina koje ćemo koristiti ćemo navesti u parametru `series`. Navodimo **sve vrijednosti** intenziteta kao string poredan od najmanjeg prema najvećem. 

<div class="alert alert-warning rounded-pill rounded-5" style="margin: auto auto 10px auto; text-align: center;">
    <strong>Napomena</strong>: Pazite da nemate razmake između brojeva!
</div>

U parametru `color_model` navodimo *labele* za intenzitete koje ćemo prikazati (bitna ova oznaka `+c`, koja označava da definiramo *labele*!).

In [ ]:
pygmt.makecpt(
    cmap="seis",
    series="",
    reverse=True,
    color_model="+c"
)

Sada crtamo intenzitete na kartu tako da zadamo da funkcija [`pygmt.Figure.plot`](https://www.pygmt.org/dev/api/generated/pygmt.Figure.plot.html) koristi vrijednosti intenziteta da pojedinoj točki dodijeli odgovarajuću boju. To ćemo postići tako da u parametar `fill` zadamo vrijednost intenziteta iz *dataframea* `df`.

Ne zaboravimo nacrtati i epicentar!

## Izoseiste

### Teorijske izoseiste prema Kövesligethy-Janosijevoj jednadžbi

Nacrtajmo sada teorijske izoseiste na kartu korištenjem Kövesligethy-Janosijeve jednadžbe:

$$I = I_{0} - 3\log\frac{r}{h} - 3\mu\alpha(r - h),$$

gdje je $I_{0}$ intenzitet u žarištu potresa, $r$ je hipocentralna udaljenost u kilometrima, $h$ je dubina žarišta potresa u kilometrima, $\mu = \log e \approx 0.4343$, a $\alpha$ je koeficijent apsorpcije makroseizmičkog intenziteta, za koji možemo pretpostaviti da je vrijednosti $\alpha=0.005$ km$^{-1}$. 

Riješimo jednadžbu numerički! Tražit ćemo rješenja funkcije $f(r)=0$. Napišimo Kövesligethy-Janosi jednadžbu kao funkciju $f(r)$.

$$f(r) = I_{0} - 3\log\frac{r}{h} - 3\mu\alpha(r - h) - I.$$


In [ ]:
import numpy as np

def f_kj(r, intensity):
    mu = np.log10(np.e)
    alpha = 0.005
    
    return i0 - 3 * np.log10(r/h) - 3 * mu * alpha * (r - h) - intensity

Rješavamo jednadžbu korištenjem [scipy.optimize.root_scalar](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.root_scalar.html) metode. 

In [ ]:
from scipy.optimize import root_scalar

r_values = list()
i_values = np.arange(i0 - 1, 0, -1)
for Int in i_values:
    sol = root_scalar(
        lambda r: f_kj(r, Int),
        bracket=[1e-6, 1e3],
        method='brentq'
    )
    r_values.append(np.sqrt(sol.root ** 2 - h ** 2))

print(r_values)

Sada ćemo jednostavno nacrtati kružnice polumjera izračunatih vrijednosti oko epicentra potresa.  Kružnice crtamo na kartu tako da definiramo elipsu s jednakom dužom i kraćom osi (za dužinu osi uzimamo **dijametar**!)

Želimo označiti izoseiste tako ta znamo koja nam označava koji intenzitet.

Uobičajeno je na kartama prikazivati intenzitete rimskim brojkama. Napisat ćemo onda jednostavnu funkciju koja će nam vraćati rimske brojeve ako joj kao argument zadajemo cjelobrojni intenzitet.

In [ ]:
def intensity2roman(value):
    if value == 1:
        return "I"

    elif value == 2:
        return "II"

    elif value == 3:
        return "III"

    elif value == 4:
        return "IV"

    elif value == 5:
        return "V"

    elif value == 6:
        return "VI"

    elif value == 7:
        return "VII"

    elif value == 8:
        return "VIII"

    elif value == 9:
        return "IX"

    elif value == 10:
        return "X"

    elif value == 11:
        return "XI"

    elif value == 12:
        return "XII"

    else:
        raise Exception("Not valid intensity value!")

Napisat ćemo iznos intenziteta na sjevernom dijelu kružnice. Za to koristimo funkciju [`pygmt.Figure.text`](https://www.pygmt.org/dev/api/generated/pygmt.Figure.text.html).

In [ ]:
for i, Int in enumerate(i_values):
    fig.text(
        x=lon,
        y=lat + kilometer2degrees(r_values[i]),
        text=intensity2roman(Int),
        font="9p,Helvetica,black",
        justify="BC",
        offset="0/0.1"
    )

fig.show()